In [ ]:
import pandas as pd
import numpy as np

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
import sqlite3

con = sqlite3.connect('../db.sqlite3')

df1 = pd.read_sql('SELECT * FROM restaurant_product', con)
df2 = pd.read_sql('SELECT * FROM restaurant_orderitem', con)
df3 = pd.read_sql('SELECT * FROM restaurant_order', con)

merged_df = df2.merge(df3, left_on='order_id', right_on='id', suffixes=('_item', '_order')) \
               .merge(df1, left_on='product_id', right_on='id', suffixes=('', '_prod'))

merged_df.to_csv('full_data.csv', index=False)

df = pd.read_csv("full_data.csv")
df.head(50)

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
top_10_product_sales = df.groupby("name")["quantity"].sum().sort_values(ascending=False).head(10)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
plt.pie(top_10_product_sales, 
        labels=top_10_product_sales.index,
        autopct='%1.1f%%',
        startangle=140)

plt.title('Top 10 Most Popular Products by Quantity')
plt.axis('equal')
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["Item Price"] = df["price"] * df["quantity"]
top_10_money = df.groupby("name")["Item Price"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 8))
plt.pie(top_10_money, 
        labels=top_10_money.index,
        autopct='%1.1f%%',
        startangle=140)

plt.title('Top 10 Most Popular Products by Item Price')
plt.axis('equal')
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df['Order Datetime'] = pd.to_datetime(df['datetime'])
df['Order Hour'] = df['Order Datetime'].dt.hour

In [ ]:
hourly_income = df.groupby('Order Hour')['Item Price'].sum()
plt.figure(figsize=(12, 6))
hourly_income.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Total Restaurant Income by Hour of Day')
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Total Income')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df['Order Day'] = df['Order Datetime'].dt.day_name()
daily_income = df.groupby('Order Day')['Item Price'].sum()
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_income = daily_income.reindex(days_order).fillna(0)

plt.figure(figsize=(10, 6))
daily_income.plot(kind='bar', color='lightgreen', edgecolor='black')

plt.title('Total Restaurant Income by Day of Week')
plt.xlabel('Day of the Week')
plt.ylabel('Total Income')
plt.xticks(rotation=45)

plt.show()